# Modul B (Rekomendasi Menu Item-to-Item) & Modul D (Klasifikasi Intent Chatbot)
**Mitra UMKM:** Banyu Bening Resto, Salatiga (`tenant_id = BBR001`)  
**Proyek Capstone:** Tumbass SaaS Operasional Restoran (ASAH / Dicoding 2026)

Notebook ini mendokumentasikan penyediaan dataset, *data wrangling*, *feature engineering*, pembagian data *train/test* bebas bocor (*anti-data-leakage*), serta evaluasi komparasi multi-model untuk:
1. **Kode B — Rekomendasi Menu (`Recommender Item-to-Item`, Halaman QR, Prioritas: Wajib):**
   - Membangun matriks ko-okurensi keranjang belanja (`order_id x menu_id`) dari 2.017 transaksi `paid`.
   - Membagi data secara kronologis (*Order-Level Chronological Split* 80% Latih : 20% Uji).
   - Membandingkan 3 metode: **M1 Global Best-Sellers (Baseline)** vs **M2 Association Rules (FP-Growth Confidence-Lift)** vs **M3 Item-to-Item Co-occurrence (Cosine Similarity + Lift + Cross-Category Bonus)** dengan metrik `Precision@3`, `Recall@3`, `HitRate@3`, `Precision@5`, `Recall@5`, dan `Catalog Coverage %`.
2. **Kode D — Klasifikasi Intent (`Klasifikasi Teks 5 Kelas`, Router Chatbot, Prioritas: Nilai Tambah):**
   - Menyediakan dataset kalimat pengguna berbahasa Indonesia untuk 5 kelas intent: `tanya_menu`, `pemesanan`, `info_resto`, `keluhan`, dan `lainnya` (termasuk *prompt injection* & *out-of-scope*).
   - Menjalankan normalisasi singkatan tidak baku dengan **mempertahankan kata negasi** serta **deduplikasi penuh sebelum pembagian data** (*Stratified 80/20 Split*).
   - Membandingkan 4 metode: **M1 Multinomial Naive Bayes**, **M2 Word TF-IDF + Logistic Regression**, **M3 Calibrated LinearSVM**, dan **M4 Hybrid Word+Char TF-IDF + Logistic Regression**.

In [1]:
import pandas as pd
from pathlib import Path

PROC = Path('../data/processed')
MODS = Path('../models')

baskets_tr = pd.read_csv(PROC / 'recommender_baskets_train.csv')
baskets_te = pd.read_csv(PROC / 'recommender_baskets_test.csv')
pairs_df = pd.read_csv(PROC / 'menu_cooccurrence_pairs.csv')
top_recs_df = pd.read_csv(PROC / 'menu_top_recommendations.csv')
intent_clean = pd.read_csv(PROC / 'intent_dataset_clean.csv')
intent_tr = pd.read_csv(PROC / 'intent_train.csv')
intent_te = pd.read_csv(PROC / 'intent_test.csv')

print('Dataset Modul B & Modul D berhasil dimuat.')
print(f'Train Baskets: {len(baskets_tr)} | Test Baskets: {len(baskets_te)}')
print(f'Pasangan Ko-okurensi Item-to-Item: {len(pairs_df)} pasangan')
print(f'Top-5 Rekomendasi Katalog: {len(top_recs_df)} baris (197 menu aktif x 5 rekomendasi)')
overlap = len(set(intent_tr['text_clean']) & set(intent_te['text_clean']))
print(f'Intent Dataset Clean: {len(intent_clean)} kalimat unik (Train: {len(intent_tr)} | Test: {len(intent_te)} | Overlap: {overlap})')

Dataset Modul B & Modul D berhasil dimuat.
Train Baskets: 1613 | Test Baskets: 404
Pasangan Ko-okurensi Item-to-Item: 4168 pasangan
Top-5 Rekomendasi Katalog: 985 baris (197 menu aktif x 5 rekomendasi)
Intent Dataset Clean: 546 kalimat unik (Train: 436 | Test: 110 | Overlap: 0)


## 1. Evaluasi Komparasi 3 Metode Modul B (Rekomendasi Menu Item-to-Item)

In [2]:
eval_b = pd.read_csv(MODS / 'evaluasi_recommender_3_metode.csv')
print('=== HASIL EVALUASI MODUL B (REKOMENDASI MENU) ===')
print(eval_b.to_string())
top_recs_df.head(10)

=== HASIL EVALUASI MODUL B (REKOMENDASI MENU) ===
                                                                                 model  test_baskets  precision_at_3  recall_at_3  hit_rate_at_3  precision_at_5  recall_at_5  hit_rate_at_5  catalog_coverage_pct
0                                        M1 Baseline: Global Best-Sellers (Popularity)           378          0.0582       0.0992         0.1746          0.0444       0.1265         0.2169                  3.05
1                         M2 Kandidat 1: Association Rules (FP-Growth Confidence-Lift)           378          0.1887       0.3408         0.4921          0.1418       0.4153         0.5661                 55.33
2  M3 Kandidat 2 (Winner): Item-to-Item Co-occurrence (Cosine + Lift + Cross-Category)           378          0.2249       0.3990         0.5794          0.1762       0.5137         0.7037                 96.45


,tenant_id,menu_id,menu_name,category_name,rank,recommended_menu_id,recommended_menu_name,recommended_category,recommended_price_idr,cosine_similarity,lift,source
0,BBR001,M0001,Air Mineral Botol,Minuman Standar,1,M0003,Ayam Bakar Tok,Alacarte,23000,0.4864,4.9569,item_to_item_hybrid
1,BBR001,M0001,Air Mineral Botol,Minuman Standar,2,M0005,Bakwan Jagung,Snack,13000,0.2829,5.8674,item_to_item_hybrid
2,BBR001,M0001,Air Mineral Botol,Minuman Standar,3,M0046,Gurame Goreng Asam Pedas,Gurame,57500,0.1634,6.1498,item_to_item_hybrid
3,BBR001,M0001,Air Mineral Botol,Minuman Standar,4,M0102,Mie Kuah Seafood,Mie,18000,0.0962,7.4676,item_to_item_hybrid
4,BBR001,M0001,Air Mineral Botol,Minuman Standar,5,M0015,Bihun Siram Sapi Spesial BP,Bihun,23000,0.1005,2.7155,item_to_item_hybrid
5,BBR001,M0002,Air Putih Gelas,Minuman Standar,1,M0004,Ayam Goreng Kremezz Tok,Alacarte,23000,0.3571,4.3758,item_to_item_hybrid
6,BBR001,M0002,Air Putih Gelas,Minuman Standar,2,M0003,Ayam Bakar Tok,Alacarte,23000,0.2178,2.2513,item_to_item_hybrid
7,BBR001,M0002,Air Putih Gelas,Minuman Standar,3,M0103,Mie Siram Ayam Spesial Blackpepper,Mie,20000,0.1543,7.6810,item_to_item_hybrid
8,BBR001,M0002,Air Putih Gelas,Minuman Standar,4,M0006,Bakwan Sayur,Snack,13000,0.1610,2.9870,item_to_item_hybrid
9,BBR001,M0002,Air Putih Gelas,Minuman Standar,5,M0017,Cha Kangkung,Sayuran,15000,0.1599,2.9461,item_to_item_hybrid


## 2. Evaluasi Komparasi 4 Metode Modul D (Klasifikasi Intent Router Chatbot)

In [3]:
eval_d = pd.read_csv(MODS / 'evaluasi_intent_4_metode.csv')
print('=== HASIL EVALUASI MODUL D (KLASIFIKASI INTENT 5 KELAS) ===')
print(eval_d.to_string())
intent_clean.groupby('intent_label')[['intent_id']].count().rename(columns={'intent_id': 'jumlah_kalimat_unik'})

=== HASIL EVALUASI MODUL D (KLASIFIKASI INTENT 5 KELAS) ===
                                                                   model  cv_5fold_macro_f1  test_accuracy  test_macro_precision  test_macro_recall  test_macro_f1
0      M2 Kandidat 1: Word TF-IDF (1,2) + Logistic Regression (balanced)             0.9917         1.0000                1.0000             1.0000         1.0000
1                M3 Kandidat 2: Word TF-IDF (1,2) + Calibrated LinearSVM             0.9917         1.0000                1.0000             1.0000         1.0000
2  M4 Kandidat 3 (Winner): Hybrid Word+Char TF-IDF + Logistic Regression             0.9891         1.0000                1.0000             1.0000         1.0000
3               M1 Baseline: Word TF-IDF (1,2) + Multinomial Naive Bayes             0.9829         0.9818                0.9845             0.9777         0.9806


,jumlah_kalimat_unik
intent_label,
info_resto,103
keluhan,103
lainnya,69
pemesanan,148
tanya_menu,123
